# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs using `mlcroissant`. Entities are referenced by their `@id` as required.

In [ ]:
# List record sets, fields, and columns by their `@id`
if hasattr(metadata, 'record_sets'):
    print("Available record sets (by @id):")
    for rs in metadata.record_sets:
        print(f"- {rs['@id']}")
        # Display fields for each record set
        if 'fields' in rs:
            print("  Fields:")
            for field in rs['fields']:
                # Each field is a dict, should have '@id'
                print(f"    - {field['@id']}")
                if 'columns' in field:
                    print(f"      Columns:")
                    for col in field['columns']:
                        print(f"        - {col['@id']}")
else:
    print("No record sets found in metadata.")

# For the FAIR^2 dataset, let's attempt to discover available record sets from the dataset implementation
print("\nAttempting to enumerate record sets directly from dataset object:")
try:
    for record_set in dataset.record_sets:
        print("-", record_set['@id'], ", name:", record_set.get('name'))
except Exception as e:
    print("Error enumerating record sets. This dataset may not expose record set metadata as expected. Error:", e)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Discover record set IDs programmatically
# If you know the record set @id, you can specify it directly, e.g.:
#   record_sets = ['cr:RecordSet', ...]
# But let's try to enumerate available record sets for this dataset:

record_sets = []
try:
    # Try to find all record set @id's
    if hasattr(dataset, 'record_sets'):
        record_sets = [rs['@id'] for rs in dataset.record_sets]
    elif hasattr(metadata, 'record_sets'):
        record_sets = [rs['@id'] for rs in metadata.record_sets]
    else:
        record_sets = []
except Exception as e:
    print(f"Could not retrieve record sets: {e}")

if not record_sets:
    # Manually specify (from FAIR^2, you might need to explore the raw metadata)
    record_sets = ['cr:RecordSet']  # Replace by the true @id as appropriate

print(f"Record sets to explore: {record_sets}")

dataframes = {}
# Try loading each record set into a dataframe
for record_set in record_sets:
    try:
        records = list(dataset.records(record_set=record_set))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set] = df
            print(f"Loaded {len(df)} records for record set '{record_set}'. Columns: {df.columns.tolist()[:6]}...")
        else:
            print(f"No records found for record set '{record_set}'.")
    except Exception as e:
        print(f"Failed to load record set '{record_set}':", e)

# If dataframes is empty, the dataset may need special access or a different record set id

# Preview first dataframe, if available
if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(f"Example columns in '{first_rs}': {dataframes[first_rs].columns.tolist()}")
    display(dataframes[first_rs].head())
else:
    print("No dataframes extracted.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
We reference each field/column by their `@id` where possible.

In [ ]:
import numpy as np
# Select a record set and a numeric field for analysis
# If you are unsure, print the DataFrame info
if dataframes:
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]
    print(f"Columns in record set '{rs_id}':")
    print(df.columns.tolist())
    # Attempt to automatically detect a numeric (float/int) column for analysis
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_fields:
        # Try to coerce columns to numbers if columns are named eg. 'MW', 'molecular_weight', etc.
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except Exception:
                continue
        numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric fields found: {numeric_fields}")

    # Let's select the first available numeric field
    if numeric_fields:
        numeric_field = numeric_fields[0]
    else:
        print("No numeric fields detected for EDA.")
        numeric_field = None

    if numeric_field:
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field if available
        group_fields = df.select_dtypes(include=[object]).columns.tolist()
        group_field = None
        for gf in group_fields:
            if df[gf].nunique() < 20 and df[gf].nunique() > 1:
                group_field = gf
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped data by {group_field}:\n{grouped_df.head()}")
        else:
            print("No suitable categorical field to group by.")
    else:
        print("No numeric field selected for EDA.")
else:
    print("No dataframe available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we show a histogram for the numeric field and a boxplot if a group field is present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field:
    plt.figure(figsize=(10, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot if group_field detected
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} distribution by {group_field}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded metadata and records from the Croissant schema using `mlcroissant`.
- We explored available record sets and fields using their `@id` references.
- We performed basic exploratory data analysis, including numeric field normalization and grouping by categorical fields.
- We visualized data distributions and groupings for deeper insights.

Further work may include more domain-specific analyses, cross-referencing fields, or building predictive models using this dataset.